# 03 Train Qwen QLoRA

This notebook fine-tunes `Qwen/Qwen2.5-7B-Instruct` as one unified conversational medical assistant using:

- QLoRA
- Unsloth
- 4-bit quantization
- TRL `SFTTrainer`

Expected input files:

- `/content/sample_data/qwen_ready/train.jsonl`
- `/content/sample_data/qwen_ready/val.jsonl`
- `/content/sample_data/qwen_ready/test.jsonl`

Expected output directory:

- `/content/medical_qwen_lora`

This notebook is optimized for Google Colab Pro with a T4 16GB GPU.

In [1]:
!pip uninstall -y transformers trl peft accelerate bitsandbytes xformers unsloth -q

!pip install -U \
transformers \
trl \
peft \
accelerate \
bitsandbytes \
datasets \
sentencepiece \
einops \
ninja

!pip install -U unsloth

!pip install -U xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Runtime Safety

Colab runtimes can disconnect or reset without warning during long fine-tuning jobs.

To reduce the chance of losing work:

- checkpoints are written to Google Drive instead of only `/content`
- training automatically resumes from the latest checkpoint when available
- logs and final artifacts are saved both locally and persistently

If the runtime crashes, reconnect the notebook and rerun the cells. The resume cell will detect the latest saved checkpoint in Drive.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ════════════════════════════════════════════════════════
# VÉRIFICATION DRIVE — à exécuter après le mount
# ════════════════════════════════════════════════════════
import os
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

DRIVE_PROJECT = Path("/content/drive/MyDrive/medical_qwen_project")
CKPT_DIR      = DRIVE_PROJECT / "checkpoints"

print("Drive project path:", DRIVE_PROJECT)
print("Exists:", DRIVE_PROJECT.exists())

if DRIVE_PROJECT.exists():
    print("\nContenu du dossier Drive:")
    for item in sorted(DRIVE_PROJECT.iterdir()):
        print(" ", item.name, "(dossier)" if item.is_dir() else f"({item.stat().st_size/1024:.1f} KB)")

if CKPT_DIR.exists():
    checkpoints = [d for d in CKPT_DIR.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
    checkpoints.sort(key=lambda d: int(d.name.split("-")[-1]))
    print(f"\nCheckpoints trouvés ({len(checkpoints)}):")
    for ck in checkpoints:
        print(" ", ck.name)
    last_ck = get_last_checkpoint(str(CKPT_DIR))
    print(f"\n✅ Dernier checkpoint : {last_ck}")
    if last_ck is None:
        print("⚠️  Aucun checkpoint — training démarrera depuis zéro")
else:
    print("⚠️  Dossier checkpoints introuvable — training démarrera depuis zéro")
    print("   Vérifiez que le dossier medical_qwen_project est partagé avec ce compte.")


In [2]:
import gc
import csv
import json
import math
import os
import random
import shutil
import statistics
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
from datasets import Dataset, DatasetDict, load_dataset
from tqdm.auto import tqdm

from transformers import EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
else:
    raise RuntimeError("A CUDA GPU is required for this notebook.")

BASE_DIR = Path("/content/sample_data/qwen_ready")
TRAIN_PATH = BASE_DIR / "train.jsonl"
VAL_PATH = BASE_DIR / "val.jsonl"
TEST_PATH = BASE_DIR / "test.jsonl"

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
LOCAL_OUTPUT_DIR = Path("/content/medical_qwen_lora")
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_SAVE_DIR = Path("/content/drive/MyDrive/medical_qwen_project")
CHECKPOINT_DIR = BASE_SAVE_DIR / "checkpoints"
FINAL_MODEL_DIR = BASE_SAVE_DIR / "final_model"
LOG_DIR = BASE_SAVE_DIR / "logs"

for directory in [BASE_SAVE_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, LOG_DIR, LOCAL_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

max_seq_length = 768
per_device_train_batch_size = 2
gradient_accumulation_steps = 4
learning_rate = 2e-4
num_train_epochs = 1
warmup_ratio = 0.05
weight_decay = 0.01
lr_scheduler_type = "cosine"
eval_steps = 200
save_steps = 200
logging_steps = 20

DEBUG_MODE = False

bf16 = False
fp16 = True

/tmp/ipykernel_6375/1432973061.py:22: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, is_bfloat16_supported


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
Total VRAM: 14.56 GB


In [3]:
import transformers
import trl
import peft

print(transformers.__version__)
print(trl.__version__)
print(peft.__version__)

5.5.0
0.24.0
0.19.1


In [4]:
@dataclass
class TrainConfig:
    model_name: str = MODEL_NAME
    output_dir: str = str(CHECKPOINT_DIR)
    local_output_dir: str = str(LOCAL_OUTPUT_DIR)
    final_model_dir: str = str(FINAL_MODEL_DIR)
    log_dir: str = str(LOG_DIR)
    max_seq_length: int = max_seq_length
    per_device_train_batch_size: int = per_device_train_batch_size
    gradient_accumulation_steps: int = gradient_accumulation_steps
    learning_rate: float = learning_rate
    num_train_epochs: int = num_train_epochs
    warmup_ratio: float = warmup_ratio
    weight_decay: float = weight_decay
    lr_scheduler_type: str = lr_scheduler_type
    seed: int = SEED
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: int = 0
    bias: str = "none"
    packing: bool = False
    assistant_only_loss: bool = False
    debug_mode: bool = DEBUG_MODE


train_config = TrainConfig()

train_config.max_seq_length = min(train_config.max_seq_length, 768)
train_config.per_device_train_batch_size = min(train_config.per_device_train_batch_size, 2)
train_config.gradient_accumulation_steps = min(train_config.gradient_accumulation_steps, 4)

print(json.dumps(asdict(train_config), indent=2))
print(f"bf16 supported: {bf16}")
print(f"fp16 fallback: {fp16}")

{
  "model_name": "Qwen/Qwen2.5-7B-Instruct",
  "output_dir": "/content/drive/MyDrive/medical_qwen_project/checkpoints",
  "local_output_dir": "/content/medical_qwen_lora",
  "final_model_dir": "/content/drive/MyDrive/medical_qwen_project/final_model",
  "log_dir": "/content/drive/MyDrive/medical_qwen_project/logs",
  "max_seq_length": 768,
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0002,
  "num_train_epochs": 1,
  "warmup_ratio": 0.05,
  "weight_decay": 0.01,
  "lr_scheduler_type": "cosine",
  "seed": 3407,
  "lora_r": 16,
  "lora_alpha": 16,
  "lora_dropout": 0,
  "bias": "none",
  "packing": false,
  "assistant_only_loss": false,
  "debug_mode": false
}
bf16 supported: False
fp16 fallback: True


In [5]:
required_files = [TRAIN_PATH, VAL_PATH, TEST_PATH]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required dataset files:\n" + "\n".join(missing))

for path in required_files:
    print(path, "->", f"{path.stat().st_size / 1024**2:.2f} MB")

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Final model directory:", FINAL_MODEL_DIR)
print("Log directory:", LOG_DIR)

/content/sample_data/qwen_ready/train.jsonl -> 60.92 MB
/content/sample_data/qwen_ready/val.jsonl -> 3.40 MB
/content/sample_data/qwen_ready/test.jsonl -> 3.40 MB
Checkpoint directory: /content/drive/MyDrive/medical_qwen_project/checkpoints
Final model directory: /content/drive/MyDrive/medical_qwen_project/final_model
Log directory: /content/drive/MyDrive/medical_qwen_project/logs


In [6]:
def validate_messages(messages):
    if not isinstance(messages, list) or len(messages) < 2:
        return False, "messages_not_valid_list"
    roles = [item.get("role") for item in messages if isinstance(item, dict)]
    if roles[:2] == []:
        return False, "missing_roles"
    for item in messages:
        if not isinstance(item, dict):
            return False, "message_not_dict"
        if item.get("role") not in {"system", "user", "assistant"}:
            return False, f"unexpected_role:{item.get('role')}"
        content = item.get("content", "")
        if not isinstance(content, str) or not content.strip():
            return False, "empty_content"
    return True, "ok"


raw_datasets = DatasetDict(
    {
        "train": load_dataset("json", data_files=str(TRAIN_PATH), split="train"),
        "validation": load_dataset("json", data_files=str(VAL_PATH), split="train"),
        "test": load_dataset("json", data_files=str(TEST_PATH), split="train"),
    }
)

validation_stats = {}
for split_name, split_ds in raw_datasets.items():
    counts = {"total": len(split_ds), "valid": 0}
    bad_examples = []
    for idx, row in enumerate(tqdm(split_ds, desc=f"Validating {split_name}")):
        ok, reason = validate_messages(row.get("messages"))
        if ok:
            counts["valid"] += 1
        elif len(bad_examples) < 5:
            bad_examples.append({"index": idx, "reason": reason})
    validation_stats[split_name] = {"counts": counts, "bad_examples": bad_examples}

    if counts["valid"] != counts["total"]:
        raise ValueError(f"{split_name} has invalid rows: {counts} ; examples={bad_examples}")

print(json.dumps(validation_stats, indent=2))
print({split: len(ds) for split, ds in raw_datasets.items()})

if min(len(ds) for ds in raw_datasets.values()) <= 0:
    raise ValueError("One of the splits is empty after loading.")

random_preview_indices = {}
for split_name, split_ds in raw_datasets.items():
    sample_count = min(3, len(split_ds))
    indices = list(range(len(split_ds)))
    random.Random(SEED + len(split_name)).shuffle(indices)
    random_preview_indices[split_name] = indices[:sample_count]

print("Random preview indices:", random_preview_indices)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Validating train:   0%|          | 0/37690 [00:00<?, ?it/s]

Validating validation:   0%|          | 0/2092 [00:00<?, ?it/s]

Validating test:   0%|          | 0/2098 [00:00<?, ?it/s]

{
  "train": {
    "counts": {
      "total": 37690,
      "valid": 37690
    },
    "bad_examples": []
  },
  "validation": {
    "counts": {
      "total": 2092,
      "valid": 2092
    },
    "bad_examples": []
  },
  "test": {
    "counts": {
      "total": 2098,
      "valid": 2098
    },
    "bad_examples": []
  }
}
{'train': 37690, 'validation': 2092, 'test': 2098}
Random preview indices: {'train': [4944, 30203, 32866], 'validation': [1137, 1839, 1292], 'test': [1686, 355, 740]}


In [7]:
print("Sample training example:")
print(json.dumps(raw_datasets["train"][0], ensure_ascii=False, indent=2)[:3000])

print("\nSample validation example:")
print(json.dumps(raw_datasets["validation"][0], ensure_ascii=False, indent=2)[:3000])

print("\nRandom sanity-check examples:")
for split_name, indices in random_preview_indices.items():
    print("\n" + "=" * 100)
    print(split_name)
    for idx in indices:
        print("-" * 100)
        print(f"Index: {idx}")
        print(json.dumps(raw_datasets[split_name][idx], ensure_ascii=False, indent=2)[:1800])

Sample training example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a careful medical assistant for educational support. Explain likely possibilities without claiming certainty, use calm natural language, and focus on practical next steps, common evaluations, and relevant self-care. When medications are mentioned, keep suggestions conservative and safety-aware, taking allergies, pregnancy, age, other medicines, and medical history into account. Do not blindly prescribe dangerous drugs, do not replace a physician, and recommend urgent care only when red-flag symptoms make it medically appropriate."
    },
    {
      "role": "user",
      "content": "my son who is 2.5 years old has high fever for last 10 days eventhough he has been administered with amoxyllin and panamex elixir on a doctors consultation.he was diagnosed with sore throat at first aand then on second visit doctor advised that it is an ear infection.please advise"
    },
    {
      "role": "

In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=train_config.model_name,
    max_seq_length=train_config.max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# IMPORTANT: Do NOT set `tokenizer.eos_token` manually — Qwen tokenizer manages EOS internally.
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("EOS token id:", getattr(tokenizer, 'eos_token_id', None))
print("special_tokens_map:", getattr(tokenizer, 'special_tokens_map', None))

# Sanity check: detect literal placeholder corruption and fail loudly (do not overwrite tokenizer.eos_token).
if isinstance(tokenizer.eos_token, str) and tokenizer.eos_token.strip() == "<EOS_TOKEN>":
    raise ValueError("Detected literal '<EOS_TOKEN>' in tokenizer.eos_token — tokenizer files appear corrupted. Do not set tokenizer.eos_token manually.")

def print_gpu_stats(stage=""):
    if not torch.cuda.is_available():
        print(f"{stage} GPU not available.")
        return
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    allocated_gb = torch.cuda.memory_allocated() / 1024**3
    reserved_gb = torch.cuda.memory_reserved() / 1024**3
    print(
        f"{stage} GPU={torch.cuda.get_device_name(0)} | "
        f"allocated={allocated_gb:.2f} GB | reserved={reserved_gb:.2f} GB | total={total_gb:.2f} GB"
    )


print_gpu_stats("After model load")

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Tokenizer vocab size: 151666
Pad token: <|im_end|>
EOS token: <|im_end|>
EOS token id: 151645
special_tokens_map: {'eos_token': '<|im_end|>', 'pad_token': '<|im_end|>'}
After model load GPU=Tesla T4 | allocated=6.70 GB | reserved=6.72 GB | total=14.56 GB


In [9]:
def render_chat(example: Dict) -> Dict[str, str]:
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


rendered_datasets = DatasetDict(
    {split: ds.map(render_chat, desc=f"Applying chat template to {split}") for split, ds in raw_datasets.items()}
)

print(rendered_datasets["train"][0]["text"][:2500])

Applying chat template to train:   0%|          | 0/37690 [00:00<?, ? examples/s]

Applying chat template to validation:   0%|          | 0/2092 [00:00<?, ? examples/s]

Applying chat template to test:   0%|          | 0/2098 [00:00<?, ? examples/s]

<|im_start|>system
You are a careful medical assistant for educational support. Explain likely possibilities without claiming certainty, use calm natural language, and focus on practical next steps, common evaluations, and relevant self-care. When medications are mentioned, keep suggestions conservative and safety-aware, taking allergies, pregnancy, age, other medicines, and medical history into account. Do not blindly prescribe dangerous drugs, do not replace a physician, and recommend urgent care only when red-flag symptoms make it medically appropriate.<|im_end|>
<|im_start|>user
my son who is 2.5 years old has high fever for last 10 days eventhough he has been administered with amoxyllin and panamex elixir on a doctors consultation.he was diagnosed with sore throat at first aand then on second visit doctor advised that it is an ear infection.please advise<|im_end|>
<|im_start|>assistant
Although tonsillitis can present with fever, sore throat and ear infection, clinical examination

In [10]:
def token_length_stats(dataset: Dataset, sample_size: int | None = 2048) -> Dict[str, float]:
    if sample_size is not None and len(dataset) > sample_size:
        indices = list(range(len(dataset)))
        random.Random(SEED).shuffle(indices)
        dataset = dataset.select(indices[:sample_size])

    lengths = []
    for row in tqdm(dataset, desc="Computing token lengths"):
        tokenized = tokenizer(
            row["text"],
            truncation=False,
            add_special_tokens=False,
        )
        lengths.append(len(tokenized["input_ids"]))

    lengths = sorted(lengths)
    if not lengths:
        raise ValueError("No token lengths computed.")

    def pct(p: float) -> int:
        idx = min(len(lengths) - 1, int((len(lengths) - 1) * p))
        return lengths[idx]

    return {
        "count": len(lengths),
        "mean": round(sum(lengths) / len(lengths), 2),
        "median": pct(0.50),
        "p90": pct(0.90),
        "p95": pct(0.95),
        "p99": pct(0.99),
        "max": lengths[-1],
        "over_768": sum(x > 768 for x in lengths),
    }


length_report = {split: token_length_stats(ds) for split, ds in rendered_datasets.items()}
print(json.dumps(length_report, indent=2))

Computing token lengths:   0%|          | 0/2048 [00:00<?, ?it/s]

Computing token lengths:   0%|          | 0/2048 [00:00<?, ?it/s]

Computing token lengths:   0%|          | 0/2048 [00:00<?, ?it/s]

{
  "train": {
    "count": 2048,
    "mean": 336.81,
    "median": 319,
    "p90": 444,
    "p95": 502,
    "p99": 705,
    "max": 1148,
    "over_768": 9
  },
  "validation": {
    "count": 2048,
    "mean": 337.26,
    "median": 319,
    "p90": 448,
    "p95": 502,
    "p99": 659,
    "max": 2648,
    "over_768": 9
  },
  "test": {
    "count": 2048,
    "mean": 336.35,
    "median": 319,
    "p90": 445,
    "p95": 501,
    "p99": 678,
    "max": 1201,
    "over_768": 11
  }
}


In [11]:
def truncate_rendered_example(example: Dict[str, str]) -> Dict[str, str]:
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=train_config.max_seq_length,
        add_special_tokens=False,
    )
    example["input_length"] = len(tokenized["input_ids"])
    return example


prepared_datasets = DatasetDict(
    {
        split: ds.map(
            truncate_rendered_example,
            desc=f"Annotating lengths for {split}",
        )
        for split, ds in rendered_datasets.items()
    }
)

if DEBUG_MODE:
    prepared_datasets = DatasetDict(
        {
            "train": prepared_datasets["train"].select(range(min(256, len(prepared_datasets["train"])))),
            "validation": prepared_datasets["validation"].select(range(min(64, len(prepared_datasets["validation"])))),
            "test": prepared_datasets["test"].select(range(min(64, len(prepared_datasets["test"])))),
        }
    )
    print("DEBUG_MODE enabled: using reduced dataset sizes.")

print(prepared_datasets)

Annotating lengths for train:   0%|          | 0/37690 [00:00<?, ? examples/s]

Annotating lengths for validation:   0%|          | 0/2092 [00:00<?, ? examples/s]

Annotating lengths for test:   0%|          | 0/2098 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'text', 'input_length'],
        num_rows: 37690
    })
    validation: Dataset({
        features: ['messages', 'text', 'input_length'],
        num_rows: 2092
    })
    test: Dataset({
        features: ['messages', 'text', 'input_length'],
        num_rows: 2098
    })
})


In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r=train_config.lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=train_config.lora_alpha,
    lora_dropout=train_config.lora_dropout,
    bias=train_config.bias,
    use_gradient_checkpointing="unsloth",
    random_state=train_config.seed,
    max_seq_length=train_config.max_seq_length,
    use_rslora=False,
)

model.print_trainable_parameters()

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [13]:
def gpu_memory_snapshot() -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {}
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    return {
        "free_gb": round(free_bytes / 1024**3, 2),
        "total_gb": round(total_bytes / 1024**3, 2),
        "allocated_gb": round(allocated, 2),
        "reserved_gb": round(reserved, 2),
    }


print(json.dumps(gpu_memory_snapshot(), indent=2))
print_gpu_stats("After LoRA setup")

{
  "free_gb": 7.54,
  "total_gb": 14.56,
  "allocated_gb": 6.85,
  "reserved_gb": 6.88
}
After LoRA setup GPU=Tesla T4 | allocated=6.85 GB | reserved=6.88 GB | total=14.56 GB


In [14]:
approx_update_steps = math.ceil(
    len(prepared_datasets["train"]) /
    (train_config.per_device_train_batch_size * train_config.gradient_accumulation_steps)
)
print("Approx optimizer update steps per epoch:", approx_update_steps)
print("Approx total update steps:", approx_update_steps * train_config.num_train_epochs)

estimated_seconds_per_step = 1.25 if DEBUG_MODE else 2.25
estimated_total_hours = (approx_update_steps * train_config.num_train_epochs * estimated_seconds_per_step) / 3600
print(f"Approx training duration estimate: {estimated_total_hours:.2f} hours")

Approx optimizer update steps per epoch: 4712
Approx total update steps: 4712
Approx training duration estimate: 2.94 hours


In [15]:
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
if last_checkpoint:
    print(f"Resuming from {last_checkpoint}")
else:
    print("No checkpoint found. Starting fresh training run.")

No checkpoint found. Starting fresh training run.


In [ ]:
import inspect
import types
from transformers import DataCollatorForSeq2Seq

# ════ CLEANUP patch précédent ════
try:
    del tokenizer.convert_tokens_to_ids
    print('✅ Cleaned up previous patch')
except AttributeError:
    pass

# ════ Checkpoint detection ════
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
print(f'Checkpoint détecté : {last_checkpoint}')

# ════ ÉTAPE 1 : Pré-tokeniser AVANT tout patch ════
print('Pre-tokenizing datasets (single process)...')

def pretokenize(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=train_config.max_seq_length,
        padding=False,
        add_special_tokens=False,
    )
    result['labels'] = [ids[:] for ids in result['input_ids']]
    return result

cols_to_keep = {'input_ids', 'attention_mask', 'labels'}
cols_to_remove = [c for c in prepared_datasets['train'].column_names if c not in cols_to_keep]

pretokenized_train = prepared_datasets['train'].map(
    pretokenize, batched=True, num_proc=1,
    remove_columns=cols_to_remove, desc='Pre-tokenizing train'
)
pretokenized_val = prepared_datasets['validation'].map(
    pretokenize, batched=True, num_proc=1,
    remove_columns=cols_to_remove, desc='Pre-tokenizing val'
)
print(f'✅ Train: {len(pretokenized_train)} ex | cols: {pretokenized_train.column_names}')

# ════ ÉTAPE 2 : Patch EOS/PAD (validation init TRL uniquement) ════
_vocab    = tokenizer.get_vocab()
_eos_id   = _vocab.get('<|im_end|>', 151645)
_orig_cvt = tokenizer.convert_tokens_to_ids
_FAKE     = {'<EOS_TOKEN>': _eos_id, '<PAD_TOKEN>': _eos_id, '<BOS_TOKEN>': _eos_id}

def _patched_convert(token):
    if isinstance(token, str) and token.strip() in _FAKE:
        return _FAKE[token.strip()]
    return _orig_cvt(token)

tokenizer.convert_tokens_to_ids = _patched_convert

# ════ ÉTAPE 3 : SFTConfig (pas de dataset_text_field) ════
warmup_steps = max(1, int(0.05 * (len(pretokenized_train) //
                (train_config.per_device_train_batch_size * train_config.gradient_accumulation_steps))))

sft_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    max_seq_length=train_config.max_seq_length,
    per_device_train_batch_size=train_config.per_device_train_batch_size,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=train_config.gradient_accumulation_steps,
    learning_rate=train_config.learning_rate,
    num_train_epochs=train_config.num_train_epochs,
    warmup_steps=warmup_steps,
    weight_decay=train_config.weight_decay,
    lr_scheduler_type=train_config.lr_scheduler_type,
    logging_steps=20,
    evaluation_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    bf16=bf16,
    fp16=fp16,
    optim='adamw_8bit',
    seed=train_config.seed,
    report_to=[],
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    group_by_length=True,
    packing=False,
    max_grad_norm=1.0,
    logging_first_step=True,
    save_safetensors=True,
    dataset_num_proc=1,
    remove_unused_columns=False,
)

_sig      = inspect.signature(SFTConfig)
_accepted = {k: v for k, v in sft_kwargs.items() if k in _sig.parameters}
training_args = SFTConfig(**_accepted)

# ════ Data collator ════
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

# ════ ÉTAPE 4 : Créer le trainer ════
_tsig = inspect.signature(SFTTrainer.__init__)
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=pretokenized_train,
    eval_dataset=pretokenized_val,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
if 'processing_class' in _tsig.parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in _tsig.parameters:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

# ════ Restaurer le tokenizer ════
try:
    del tokenizer.convert_tokens_to_ids
except AttributeError:
    pass

print('\n✅ Trainer créé avec succès !')
print(f'Checkpoint de reprise : {last_checkpoint}')
print(f'Steps total : {training_args.max_steps} | Epochs : {training_args.num_train_epochs}')


In [ ]:
# ════════════════════════════════════════════════════
# TRAINING — reprend automatiquement depuis le dernier checkpoint
# ════════════════════════════════════════════════════

# Fix: TRL 0.24 + Unsloth incompatibilité entropy_from_logits
def _simple_compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    outputs = model(**inputs)
    loss = outputs.loss
    return (loss, outputs) if return_outputs else loss

trainer.compute_loss = types.MethodType(_simple_compute_loss, trainer)

start_time = time.time()

def gpu_memory_snapshot():
    if torch.cuda.is_available():
        free  = torch.cuda.mem_get_info()[0] / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        alloc = torch.cuda.memory_allocated() / 1024**3
        res   = torch.cuda.memory_reserved() / 1024**3
        return {'free_gb': round(free,2), 'total_gb': round(total,2),
                'allocated_gb': round(alloc,2), 'reserved_gb': round(res,2)}
    return {}

def print_gpu_stats(label=''):
    if torch.cuda.is_available():
        n = torch.cuda.get_device_name(0)
        a = torch.cuda.memory_allocated()/1024**3
        r = torch.cuda.memory_reserved()/1024**3
        t = torch.cuda.get_device_properties(0).total_memory/1024**3
        print(f'{label} GPU={n} | allocated={a:.2f} GB | reserved={r:.2f} GB | total={t:.2f} GB')

before_train_mem = gpu_memory_snapshot()
print('GPU memory before training:', before_train_mem)
print_gpu_stats('Before training')

try:
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower() or 'cuda oom' in str(exc).lower():
        torch.cuda.empty_cache()
        gc.collect()
        print('CUDA OOM encountered during training.')
    raise

elapsed_minutes = round((time.time() - start_time) / 60, 2)
after_train_mem = gpu_memory_snapshot()
print('\nTraining finished.')
print(f'Elapsed: {elapsed_minutes} min')
print('GPU after training:', after_train_mem)
print_gpu_stats('After training')
print(train_result)


In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

# Retire le callback Colab qui bloque l'éval standalone
trainer.remove_callback(NotebookProgressCallback)

# Pre-tokenize le test set
cols_to_remove_test = [c for c in prepared_datasets["test"].column_names
                       if c not in {"input_ids", "attention_mask", "labels"}]
pretokenized_test = prepared_datasets["test"].map(
    pretokenize, batched=True, num_proc=1,
    remove_columns=cols_to_remove_test,
    desc="Pre-tokenizing test"
)

# Évaluation
eval_metrics = trainer.evaluate(pretokenized_val)
eval_loss = float(eval_metrics["eval_loss"])
eval_perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")

test_metrics = trainer.evaluate(pretokenized_test, metric_key_prefix="test")
test_loss = float(test_metrics["test_loss"])
test_perplexity = math.exp(test_loss) if test_loss < 20 else float("inf")

print(
    json.dumps(
        {
            "validation": {**eval_metrics, "perplexity": eval_perplexity},
            "test": {**test_metrics, "perplexity": test_perplexity},
        },
        indent=2,
        default=float,
    )
)


In [ ]:
trainer.save_model(str(LOCAL_OUTPUT_DIR))
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(LOCAL_OUTPUT_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

if getattr(model, "generation_config", None) is not None:
    model.generation_config.save_pretrained(str(LOCAL_OUTPUT_DIR))
    model.generation_config.save_pretrained(str(FINAL_MODEL_DIR))

config_payload = asdict(train_config)
config_payload["base_dir"] = str(BASE_DIR)
config_payload["model_name"] = MODEL_NAME
config_payload["length_report"] = length_report
config_payload["validation_stats"] = validation_stats
config_payload["bf16"] = bf16
config_payload["fp16"] = fp16

for save_dir in [LOCAL_OUTPUT_DIR, FINAL_MODEL_DIR]:
    with (save_dir / "training_config.json").open("w", encoding="utf-8") as handle:
        json.dump(config_payload, handle, ensure_ascii=False, indent=2)

metrics_payload = {
    "validation": {**eval_metrics, "perplexity": eval_perplexity},
    "test": {**test_metrics, "perplexity": test_perplexity},
}
for save_dir in [LOCAL_OUTPUT_DIR, FINAL_MODEL_DIR]:
    with (save_dir / "eval_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(metrics_payload, handle, ensure_ascii=False, indent=2, default=float)

log_history = trainer.state.log_history
with (LOG_DIR / "training_log_history.json").open("w", encoding="utf-8") as handle:
    json.dump(log_history, handle, ensure_ascii=False, indent=2, default=float)

csv_rows = []
for row in log_history:
    flat_row = {key: value for key, value in row.items() if isinstance(value, (int, float, str))}
    csv_rows.append(flat_row)

fieldnames = sorted({key for row in csv_rows for key in row.keys()})
with (LOG_DIR / "training_log_history.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for row in csv_rows:
        writer.writerow(row)

with (LOG_DIR / "summary_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics_payload, handle, ensure_ascii=False, indent=2, default=float)

print("Saved local artifacts to:", LOCAL_OUTPUT_DIR)
print("Saved persistent artifacts to:", FINAL_MODEL_DIR)
print("Saved logs to:", LOG_DIR)
print(sorted(p.name for p in FINAL_MODEL_DIR.iterdir()))

In [ ]:
FastLanguageModel.for_inference(model)

generation_examples = [
    {
        "name": "Symptom discussion",
        "messages": [
            {"role": "system", "content": "You are a careful medical assistant for educational support."},
            {"role": "user", "content": "I've had fever, sore throat, and body aches for two days. What could this be and what can I do at home?"},
        ],
    },
    {
        "name": "Illness explanation",
        "messages": [
            {"role": "system", "content": "You are a careful medical assistant for educational support."},
            {"role": "user", "content": "Can you explain what migraines are in simple language and what usually triggers them?"},
        ],
    },
    {
        "name": "Medication question",
        "messages": [
            {"role": "system", "content": "You are a careful medical assistant for educational support."},
            {"role": "user", "content": "Is ibuprofen usually okay for mild muscle pain, and when should someone avoid it?"},
        ],
    },
    {
        "name": "Emergency symptom case",
        "messages": [
            {"role": "system", "content": "You are a careful medical assistant for educational support."},
            {"role": "user", "content": "My father has chest pressure, sweating, and shortness of breath that started suddenly. What should we do right now?"},
        ],
    },
    {
        "name": "Conversational follow-up",
        "messages": [
            {"role": "system", "content": "You are a careful medical assistant for educational support."},
            {"role": "user", "content": "I've had stomach pain after meals for a week. What else would you want to know before narrowing it down?"},
        ],
    },
    {
        "name": "Mild symptoms",
        "messages": [
            {"role": "system", "content": "You are a careful medical assistant for educational support."},
            {"role": "user", "content": "I have a mild headache and a runny nose since yesterday. What are some safe home measures I can try?"},
        ],
    },
    {
        "name": "Nonsense input",
        "messages": [
            {"role": "system", "content": "You are a careful medical assistant for educational support."},
            {"role": "user", "content": "My refrigerator is coughing and my shadow has diabetes. What does that mean?"},
        ],
    },
]


def generate_reply(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=prompt,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion = outputs[0][prompt.shape[-1]:]
    return tokenizer.decode(completion, skip_special_tokens=True).strip()


for example in generation_examples:
    print("\n" + "=" * 100)
    print(example["name"])
    print("-" * 100)
    print("User:", example["messages"][-1]["content"])
    try:
        reply = generate_reply(example["messages"])
        print("Assistant:", reply)
    except RuntimeError as exc:
        print("Generation failed:", exc)
        torch.cuda.empty_cache()

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

# Retire le callback Colab qui bloque l'éval standalone
trainer.remove_callback(NotebookProgressCallback)

# Pre-tokenize le test set
cols_to_remove_test = [c for c in prepared_datasets['test'].column_names
                        if c not in {'input_ids', 'attention_mask', 'labels'}]
pretokenized_test = prepared_datasets['test'].map(
    pretokenize, batched=True, num_proc=1,
    remove_columns=cols_to_remove_test, desc='Pre-tokenizing test'
)

eval_metrics = trainer.evaluate(pretokenized_val)
eval_loss = float(eval_metrics['eval_loss'])
eval_perplexity = math.exp(eval_loss) if eval_loss < 20 else float('inf')

test_metrics = trainer.evaluate(pretokenized_test, metric_key_prefix='test')
test_loss = float(test_metrics['test_loss'])
test_perplexity = math.exp(test_loss) if test_loss < 20 else float('inf')

print(json.dumps({
    'validation': {**eval_metrics, 'perplexity': eval_perplexity},
    'test': {**test_metrics, 'perplexity': test_perplexity},
}, indent=2, default=float))


## Notes

- This notebook fine-tunes one unified conversational model rather than separate NLU and NLG modules.
- `max_seq_length = 768` is intentional for T4 stability.
- If training is interrupted, rerun the notebook and it will resume from the latest checkpoint in Google Drive when available.
- After training, you can optionally merge LoRA weights later on a larger GPU if you want a standalone merged model.